In [1]:
import json
import torch         
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate

from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    pipeline,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)

/home/majid/Desktop/MS2/data/110/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset
je me suis basé sur le generate_dataset de maxime
j'ai ajouté des cas négatifs et des cas ambigus
case friendly 

## LOAD DATA
also load the tokenizer. 
use camembert-base tokenizer : neutre 

ici on change le path pour le dataset

In [2]:

with open("../data/dataset_ner_cities_split_essai1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

train_dataset = Dataset.from_list(data["train"])
val_dataset   = Dataset.from_list(data["val"])

train_dataset = train_dataset.remove_columns(
    ["sentence", "start_city", "end_city"]
)
val_dataset = val_dataset.remove_columns(
    ["sentence", "start_city", "end_city"]
)


## Evaluate
Evaluate_model inspiree de evaluate de modelBuilder de Maxime pour la consistency entre collègues, adaptée à la pipeline 

In [3]:

def evaluate_model(model, tokenizer, dataset, label_list, data_collator):
    """
    Evaluation complète d'un modèle déjà entraîné (compatible Trainer).
    """

    print("=" * 70)
    print("CHARGEMENT DU MODELE")
    print("=" * 70)

    metric = evaluate.load("seqeval")

    def compute_metrics_for_eval(predictions, labels):
        predictions = np.argmax(predictions, axis=2)

        true_predictions = [
            [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
            for pred, lab in zip(predictions, labels)
        ]
        true_labels = [
            [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
            for pred, lab in zip(predictions, labels)
        ]

        results = metric.compute(
            predictions=true_predictions,
            references=true_labels
        )

        return results, true_predictions, true_labels

    trainer = Trainer(
        model=model,
        eval_dataset=dataset,
        data_collator=data_collator,
    )

    print("\nEVALUATION EN COURS...")
    predictions, labels, metrics = trainer.predict(dataset)

    seqeval_results, true_preds, true_labels = compute_metrics_for_eval(
        predictions, labels
    )

    print("\nSeqEval Results:")
    print(seqeval_results)

    # =====================================================
    # FLATTEN POUR MATRICE DE CONFUSION TOKEN LEVEL
    # =====================================================

    flat_preds = []
    flat_labels = []

    for preds, labs in zip(true_preds, true_labels):
        for p, l in zip(preds, labs):
            flat_preds.append(p)
            flat_labels.append(l)

    print("\nClassification Report (token-level)")
    print(classification_report(flat_labels, flat_preds, zero_division=0))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    cm = confusion_matrix(flat_labels, flat_preds, labels=label_list)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap="Blues",
        xticklabels=label_list,
        yticklabels=label_list
    )
    plt.xlabel("Predictions")
    plt.ylabel("True Labels")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=300)
    plt.close()

    print("confusion_matrix.png sauvegardé")
    
    # =====================================================
    # METRICS PAR ENTITE (seqeval entity-level)
    # =====================================================
    
    f1_scores = []
    precisions = []
    recalls = []
    
    # Extraire les types d'entités sans BIO
    entity_types = sorted(
        set(l.split("-")[1] for l in label_list if l != "O")
    )
    
    for ent in entity_types:
        if ent in seqeval_results:
            f1_scores.append(float(seqeval_results[ent]["f1"]))
            precisions.append(float(seqeval_results[ent]["precision"]))
            recalls.append(float(seqeval_results[ent]["recall"]))
        else:
            f1_scores.append(0.0)
            precisions.append(0.0)
            recalls.append(0.0)
    
    x = np.arange(len(entity_types))
    width = 0.25
    
    plt.figure(figsize=(14, 7))
    plt.bar(x - width, precisions, width, label="Precision")
    plt.bar(x, recalls, width, label="Recall")
    plt.bar(x + width, f1_scores, width, label="F1")
    
    plt.xticks(x, entity_types)
    plt.ylim(0, 1.1)
    plt.legend()
    plt.title("Metrics per Entity (Entity-level)")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("metrics_per_entity.png", dpi=300)
    plt.close()
    
    print("metrics_per_entity.png sauvegardé")

    # =====================================================
    # METRICS GLOBAL
    # =====================================================

    global_metrics = {
        "precision": seqeval_results["overall_precision"],
        "recall": seqeval_results["overall_recall"],
        "f1": seqeval_results["overall_f1"],
        "accuracy": seqeval_results["overall_accuracy"],
    }

    with open("metrics_results.json", "w", encoding="utf-8") as f:
        json.dump(global_metrics, f, indent=2)

    print("metrics_results.json sauvegardé")

    print("\nEvaluation terminée.")
    print("=" * 70)

    return global_metrics

ERROR! Session/line number was not unique in database. History logging moved to new session 6


# Train and save 

### !!! les zeros sont dominants dans une phrase 

Les poids correspondent à label_list = ["O", "B-DEP", "I-DEP", "B-ARR", "I-ARR"]

                  index :    0      1      2      3      4

 - **O est dominant → on lui donne un poids faible (0.1)**

 - Les entités sont rares → poids élevé (1.0)

 - B- sont plus importants que I- car ils marquent le début → on peut les booster légèrement

O     → 0.1   (très dominant, on pénalise peu ses erreurs)

B-DEP → 1.5   (rare + important : début d'entité DEP)

I-DEP → 1.0   (rare mais moins critique que le B-)

B-ARR → 1.5   (même logique que B-DEP)

I-ARR → 1.0

solution --> 

**WEIGHTED TRAINER** 


In [8]:
CLASS_WEIGHTS = [0.1, 1.5, 1.0, 1.5, 1.0]
#                 O  B-DEP I-DEP B-ARR I-ARR

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = torch.tensor(CLASS_WEIGHTS, device=logits.device, dtype=torch.float)

        loss_fct = nn.CrossEntropyLoss(
            weight=weights,
            ignore_index=-100   # ignore les tokens spéciaux (<s>, </s>, padding)
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [4]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [5]:
CHECKPOINT = "camembert-base"   # source unique
MODEL_PATH  = "./camembert-ner-essai1"  # où on sauvegarde

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

# Vérifie que c'est bien le bon tokenizer
print("Vocab size :", tokenizer.vocab_size)  # doit afficher 32005



Vocab size : 32005


## Configurations pour entrainement 

In [6]:
# =====================================================
# LABELS
# =====================================================

label_list = ["O", "B-DEP", "I-DEP", "B-ARR", "I-ARR"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# =====================================================
# MODEL
# =====================================================

model = AutoModelForTokenClassification.from_pretrained(
    "camembert-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob = 0.3,
    attention_probs_dropout_prob = 0.3
)


# =====================================================
# DATA COLLATOR
# =====================================================

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)



Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 8269.56it/s]
CamembertForTokenClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:

# TRAINING ARGUMENTS 
# =====================================================

training_args = TrainingArguments(
    output_dir="./camembert-ner-essai1",
    learning_rate=2e-5,                
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,                
    weight_decay=0.01,                 
    warmup_ratio=0.06,                 
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",        
    greater_is_better=True,
    save_total_limit=2                 
)

# TRAINER
# =====================================================

trainer = WeightedTrainer(         
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# TRAIN


In [ ]:
trainer.train()

trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

# import and test 


In [10]:

model_loaded    = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
tokenizer_loaded = AutoTokenizer.from_pretrained(MODEL_PATH)

ner_pipe = pipeline(
    "token-classification",
    model=model_loaded,
    tokenizer=tokenizer_loaded,
    aggregation_strategy="first"
)

phrases_test = [
    "je veux aller de Lyon à Paris",
    "Clermont-Ferrand à Lille",
    "je vais à Bordeaux",
    "il fait beau aujourd'hui",
    "moi et mon ami Lyon avons marre de Nantes et voulons aller à Saint-Étienne",
    "je pars de Saint-Étienne pour aller à Aix-en-Provence",
]

print("\n" + "=" * 60)
for text in phrases_test:
    results = ner_pipe(text)
    dep = next((r for r in results if r["entity_group"] == "DEP"), None)
    arr = next((r for r in results if r["entity_group"] == "ARR"), None)
    print(f"\n '{text}'")
    print(f"   DEP : {dep['word'] if dep else '—'}")
    print(f"   ARR : {arr['word'] if arr else '—'}")
print("=" * 60)

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 3713.66it/s]




 'je veux aller de Lyon à Paris'
   DEP : Lyon
   ARR : Paris

 'Clermont-Ferrand à Lille'
   DEP : Clermont-Ferrand
   ARR : Lille

 'je vais à Bordeaux'
   DEP : —
   ARR : Bordeaux

 'il fait beau aujourd'hui'
   DEP : —
   ARR : —

 'moi et mon ami Lyon avons marre de Nantes et voulons aller à Saint-Étienne'
   DEP : Nantes
   ARR : Saint-Étienne

 'je pars de Saint-Étienne pour aller à Aix-en-Provence'
   DEP : Saint-Étienne
   ARR : Aix-en-Provence


/home/majid/Desktop/MS2/data/110/venv/lib/python3.12/site-packages/transformers/pipelines/token_classification.py:444: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


In [12]:
evaluate_model(model_loaded, tokenizer_loaded, val_dataset, label_list, data_collator)

CHARGEMENT DU MODELE

EVALUATION EN COURS...


/home/majid/Desktop/MS2/data/110/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



SeqEval Results:
{'ARR': {'precision': np.float64(0.9875712066286898), 'recall': np.float64(0.995822454308094), 'f1': np.float64(0.9916796671866874), 'number': np.int64(1915)}, 'DEP': {'precision': np.float64(0.9882736156351791), 'recall': np.float64(0.9954068241469817), 'f1': np.float64(0.99182739457339), 'number': np.int64(1524)}, 'overall_precision': np.float64(0.9878822850548182), 'overall_recall': np.float64(0.9956382669380633), 'overall_f1': np.float64(0.991745112237509), 'overall_accuracy': 0.9987733602713038}

Classification Report (token-level)
              precision    recall  f1-score   support

       B-ARR       0.99      1.00      1.00      1915
       B-DEP       0.99      1.00      1.00      1524
       I-ARR       1.00      1.00      1.00      6838
       I-DEP       1.00      1.00      1.00      5389
           O       1.00      1.00      1.00     12052

    accuracy                           1.00     27718
   macro avg       1.00      1.00      1.00     27718
weigh

{'precision': np.float64(0.9878822850548182),
 'recall': np.float64(0.9956382669380633),
 'f1': np.float64(0.991745112237509),
 'accuracy': 0.9987733602713038}